# Trabajo Práctico N° 1: Redes Neuronales CBOW para PLN
**Asignatura**: Aprendizaje Automático Avanzado  
**Carrera**: Tecnicatura Universitaria en Inteligencia Artificial  
**Institución**: Universidad Nacional de Hurlingham (UNAHUR)  
**Profesor**: Juan Miguel Santos (LIDEC)  

---

## Descripción del Proyecto
Este Jupyter Notebook implementa el modelo de representación contextual de palabras **CBOW (Continuous Bag-of-Words)** utilizando **Python y CuPy GPU / PyTorch GPU** para aceleración en placas de video NVIDIA.

La implementación se adhiere estrictamente al material teórico y a la formulación matemática expuesta por la cátedra:
- **Entrada**: Contexto de $C = 2 \times W$ palabras ($W$ a la izquierda, $W$ a la derecha).
- **Capa Oculta**: Activación lineal $h = \frac{1}{C} \sum_{c=1}^C W_{I_c, :}^T$
- **Activación y Salida**: Softmax Completa $O(|V|)$ o **Muestreo Negativo (Negative Sampling)** con sigmoides logísticas binarias sobre la distribución $U^{3/4}$.
- **Actualización de Gradientes**: Vectorizada en VRAM GPU sin bucles secuenciales en Python.
- **Token Especial**: Únicamente `<UNK>`.
- **Selección de Vocabulario**: Elección excluyente entre `cantidad_palabras_unicas` O `porcentaje_palabras_unicas` con muestreo aleatorio reproducible (`semilla_aleatoria = 26`).
- **Reanudación por Épocas Adicionales**: Método `entrenar_adicional(epocas_adicionales=N)`.

## Sección 1: Configuración de Parámetros y Entorno

La configuración del proyecto se gestiona de forma interactiva y centralizada mediante el **Panel de Control de Hiperparámetros** a continuación.

Al ejecutar la celda de código, los valores definidos en el diccionario `PARAMETROS_PANEL` se aplicarán al objeto `config` en memoria y **se autoguardarán automáticamente** en el archivo físico [`configuracion.yaml`](configuracion.yaml) en disco, garantizando una experimentación ágil y la sincronización total con scripts futuros o ejecuciones externas.

### Parámetros Principales Configurados:
- **`ruta_corpus`**: Ruta al archivo del corpus de texto (ej. `datos/corpus.txt`).
- **`motor_computo`**: Motor de ejecución acelerada en GPU NVIDIA (`"cupy"` o `"pytorch"`).
- **`criterio_seleccion_vocabulario`**: Criterio excluyente (`"cantidad"` o `"porcentaje"`).
- **`cantidad_palabras_unicas` / `porcentaje_palabras_unicas`**: Tamaño del vocabulario seleccionado al azar con semilla `26`.
- **`muestreo_negativo`**: `True` para Muestreo Negativo $O(K)$, `False` para Softmax Completa $O(|V|)$.
- **`cantidad_muestras_negativas`**: Cantidad $K$ de palabras negativas a muestrear ($5$).
- **`tamanio_ventana`**: Ventana $W$ de palabras a izquierda y derecha ($W = 4$).
- **`dimension_embedding`**: Dimensión de la capa oculta ($N = 100$).
- **`tasa_aprendizaje`**: Tasa de aprendizaje $\eta = 0.2$.
- **`cantidad_epocas`**: Número de épocas a entrenar de cero o adicionales a sumar si se reanuda.
- **`reanudar_entrenamiento`**: `False` para entrenamiento limpio desde época 0, `True` para reanudar desde checkpoint.
- **`ruta_checkpoint`**: Ruta al archivo `.npz` de resguardo desde el cual reanudar.
- **`hacer_respaldo`**: Booleano que determina si se activan (`True`) o desactivan (`False`) los respaldos automáticos.
- **`frecuencia_respaldo`**: Frecuencia en épocas entre checkpoints ($2$). Requiere `hacer_respaldo=True` y debe ser un entero mayor a 0.
- **`tamanio_lote`**: Tamaño de mini-lote $B$ para procesamiento vectorizado en GPU ($2048$).

In [ ]:
import sys
from pathlib import Path

# Asegurar acceso a los módulos del directorio codigo/
ruta_proyecto = Path.cwd()
if str(ruta_proyecto) not in sys.path:
    sys.path.append(str(ruta_proyecto))

from codigo.configuracion import Configuracion
from codigo.tokenizador import Tokenizador
from codigo.generador_vocabulario import GeneradorVocabulario
from codigo.modelo_cbow_cupy import ModeloCbowCuPy, USAR_CUPY
from codigo.modelo_cbow_pytorch import ModeloCbowPyTorch
from codigo.entrenador_cbow import EntrenadorCbow
from codigo.evaluador_similaridad import EvaluadorSimilaridad

# ==============================================================================
# PANEL DE CONTROL DE HIPERPARÁMETROS (MODELO CBOW)
# Modifique libremente los valores en este diccionario para sus experimentos.
# Al ejecutar esta celda, los cambios se guardarán automáticamente en configuracion.yaml
# ==============================================================================
PARAMETROS_PANEL = {
    # Motor de Cómputo
    "motor_computo": "pytorch",                      # Opciones: "cupy" o "pytorch"
    
    # Corpus y Vocabulario
    "ruta_corpus": "datos/corpus.txt",
    "estrategia_tokenizacion": "palabra",            # Opciones: "palabra" o "bpe"
    "token_desconocido": "<UNK>",
    "criterio_seleccion_vocabulario": "porcentaje",   # Opciones: "cantidad" o "porcentaje"
    "cantidad_palabras_unicas": 15000,
    "porcentaje_palabras_unicas": 1.0,
    "frecuencia_minima": 1,
    
    # Hiperparámetros CBOW
    "tamanio_ventana": 4,                           # Ventana W (izquierda/derecha)
    "dimension_embedding": 100,                     # Dimensión N
    "tasa_aprendizaje": 0.2,                         # Tasa de aprendizaje eta
    "cantidad_epocas": 10,                          # Épocas a entrenar de cero o adicionales a sumar
    "tamanio_lote": 2048,                           # Muestras por lote (GPU 4-6GB: 1024-2048 | 8-12GB: 4096-8192 | 16GB+: 8192-16384)
    
    # Control de Reanudación (SSoT)
    "reanudar_entrenamiento": False,                # False: entrenamiento limpio desde época 0 | True: reanudar desde checkpoint
    "ruta_checkpoint": "respaldos/modelo_cbow_w4_epoca_1000.npz",  # Ruta al checkpoint para reanudar
    
    # Muestreo Negativo (Negative Sampling)
    "muestreo_negativo": False,                     # True: K binarias, False: Softmax global
    "cantidad_muestras_negativas": 5,              # Cantidad K de palabras negativas
    
    # Resguardos
    "directorio_respaldos": "respaldos",
    "hacer_respaldo": False,                        # True: activa autoguardado, False: desactiva respaldos
    "frecuencia_respaldo": 2,                       # Autoguardado cada K épocas (requiere hacer_respaldo=True, entero > 0)
    "semilla_aleatoria": 26,
}

# --- MECANISMO DE SINCRONIZACIÓN Y AUTOGUARDADO ---
# 1. Cargar la instancia de configuración centralizada
config = Configuracion("configuracion.yaml")

# 2. Actualizar los atributos del objeto en memoria según el Panel de Control
for clave, valor in PARAMETROS_PANEL.items():
    setattr(config, clave, valor)

# 3. Autoguardar instantáneamente en el archivo físico configuracion.yaml en disco
config.guardar_en_yaml()

# 4. Mostrar reporte de configuración sincronizada
print("=== Panel de Control: Hiperparámetros Sincronizados con configuracion.yaml ===")
diccionario_params = config.a_diccionario()
for clave, valor in diccionario_params.items():
    print(f" - {clave}: {valor}")

print(f"\nAceleración GPU (CuPy disponible): {USAR_CUPY}")

## Sección 2: Carga y Preprocesamiento del Corpus
Cargamos el archivo de texto especificado en `ruta_corpus` (`datos/corpus.txt`) y realizamos la tokenización por palabras utilizando expresiones regulares (respetando minúsculas, tildes y caracteres en español).

In [ ]:
# Instanciar tokenizador
tokenizador = Tokenizador(estrategia=config.estrategia_tokenizacion)
ruta_corpus = Path(config.ruta_corpus)

print(f"Leyendo y tokenizando corpus desde: {ruta_corpus}...")
tokens_corpus = tokenizador.tokenizar_archivo(ruta_corpus)
print(f"Total de palabras/tokens en el corpus: {len(tokens_corpus):,}")
print(f"Muestra de primeros 20 tokens: {tokens_corpus[:20]}")

## Sección 3: Construcción del Vocabulario y Distribución $U^{3/4}$
Construimos el vocabulario seleccionando palabras al azar (semilla 26) y calculamos la distribución de frecuencia unigrama rebalanceada a la potencia $3/4$ ($P(w) \propto f(w)^{0.75}$) para el Muestreo Negativo.

Al finalizar la construcción, el vocabulario generado se autoguarda de forma transparente en `respaldos/vocabulario.json` utilizando los métodos nativos de nuestro módulo de vocabulario.

In [ ]:
# Instanciar generador de vocabulario
vocabulario = GeneradorVocabulario(token_desconocido=config.token_desconocido)
ruta_vocabulario_backup = Path(config.directorio_respaldos) / "vocabulario.json"

if config.reanudar_entrenamiento:
    print(f"Modo Reanudación Activado: Cargando vocabulario guardado desde '{ruta_vocabulario_backup}'...")
    if not ruta_vocabulario_backup.exists():
        raise FileNotFoundError(
            f"Error crítico: Se solicitó reanudar entrenamiento (reanudar_entrenamiento=True), "
            f"pero el archivo de vocabulario '{ruta_vocabulario_backup}' no existe en disco."
        )
    vocabulario.cargar_vocabulario(ruta_vocabulario_backup)
    print(f"Vocabulario cargado exitosamente ({vocabulario.tamanio_vocabulario:,} palabras).")
else:
    print("Modo Entrenamiento Limpio: Construyendo vocabulario desde el corpus...")
    vocabulario.construir_vocabulario(
        lista_tokens=tokens_corpus,
        criterio_seleccion=config.criterio_seleccion_vocabulario,
        cantidad_palabras_unicas=config.cantidad_palabras_unicas,
        porcentaje_palabras_unicas=config.porcentaje_palabras_unicas,
        frecuencia_minima=config.frecuencia_minima,
        semilla_aleatoria=config.semilla_aleatoria,
    )
    # Autoguardar vocabulario para futuros resguardos/reanudaciones
    vocabulario.guardar_vocabulario(ruta_vocabulario_backup)
    print(f"Vocabulario guardado automáticamente en '{ruta_vocabulario_backup}'.")

# Convertir la secuencia de palabras del corpus a una lista de índices enteros
indices_corpus = vocabulario.convertir_tokens_a_indices(tokens_corpus)
print(f"Total de índices procesados: {len(indices_corpus):,}")

# Obtener la distribución de probabilidad unigrama P(w)^0.75 condicional al Muestreo Negativo
if config.muestreo_negativo:
    distribucion_unigrama = vocabulario.obtener_distribucion_unigrama(exponente=0.75)
    print(f"Muestreo Negativo activado: Distribución unigrama P(w)^0.75 calculada ({len(distribucion_unigrama)} elementos).")
else:
    distribucion_unigrama = None
    print("Muestreo Negativo desactivado (Softmax Completa): Se omite el cálculo de la distribución unigrama P(w)^0.75.")

## Sección 4: Celda de Ejecución Pura de Entrenamiento CBOW

Esta celda ejecuta el entrenamiento del modelo CBOW utilizando directamente los parámetros configurados en el objeto centralizado `config` (`config.reanudar_entrenamiento`, `config.ruta_checkpoint`, `config.cantidad_epocas`, `config.motor_computo`, etc.).

- **Si `config.reanudar_entrenamiento` es `True`**: Carga el estado del modelo desde el archivo `.npz` configurado en `config.ruta_checkpoint` y ejecuta `entrenador.entrenar_adicional(..., epocas_adicionales=config.cantidad_epocas)`.
- **Si `config.reanudar_entrenamiento` es `False`**: Inicializa una instancia limpia del modelo y entrena desde la época 0 por `config.cantidad_epocas` épocas.

Las instancias resultantes quedan asignadas a las variables globales unificadas `modelo_cbow`, `entrenador` e `historial_perdida`.

In [ ]:
from pathlib import Path

# --- 1. GARANTÍA DE CONSISTENCIA Y SINCRONIZACIÓN EN DISCO ---
if config.hacer_respaldo:
    if not isinstance(config.frecuencia_respaldo, int) or config.frecuencia_respaldo <= 0:
        raise ValueError(
            f"Configuración inválida: 'hacer_respaldo' es True, pero 'frecuencia_respaldo' ({config.frecuencia_respaldo}) debe ser un entero mayor que 0."
        )

# Sincronizar físicamente en configuracion.yaml en disco
config.guardar_en_yaml()

print("=== Sección 4: Ejecución Pura de Entrenamiento CBOW ===")
print(f" - Modo: {'Reanudación desde Checkpoint' if config.reanudar_entrenamiento else 'Entrenamiento Limpio desde Época 0'}")
if config.reanudar_entrenamiento:
    print(f" - Ruta Checkpoint: {config.ruta_checkpoint}")
print(f" - Motor de Cómputo: {config.motor_computo}")
print(f" - Tamaño de Lote: {config.tamanio_lote}")
print(f" - Tasa de Aprendizaje: {config.tasa_aprendizaje}")
print(f" - Épocas a Procesar: {config.cantidad_epocas}")
print(f" - Resguardos: {'Activados (cada ' + str(config.frecuencia_respaldo) + ' épocas)' if config.hacer_respaldo else 'Desactivados'}\n")

# --- 2. LÓGICA DE ENTRENAMIENTO BIFURCADA ---
if config.reanudar_entrenamiento:
    ruta_checkpoint_path = Path(config.ruta_checkpoint)
    if not ruta_checkpoint_path.exists():
        raise FileNotFoundError(
            f"Error crítico: No se encontró el archivo de checkpoint '{config.ruta_checkpoint}' para reanudar el entrenamiento."
        )
    
    # Instanciación dinámica del modelo según motor configurado
    if config.motor_computo == "pytorch":
        modelo_cbow = ModeloCbowPyTorch(
            tamanio_vocabulario=vocabulario.tamanio_vocabulario,
            dimension_embedding=config.dimension_embedding,
            muestreo_negativo=config.muestreo_negativo,
            cantidad_muestras_negativas=config.cantidad_muestras_negativas,
            distribucion_unigrama=distribucion_unigrama,
            semilla_aleatoria=config.semilla_aleatoria,
        )
    else:
        modelo_cbow = ModeloCbowCuPy(
            tamanio_vocabulario=vocabulario.tamanio_vocabulario,
            dimension_embedding=config.dimension_embedding,
            muestreo_negativo=config.muestreo_negativo,
            cantidad_muestras_negativas=config.cantidad_muestras_negativas,
            distribucion_unigrama=distribucion_unigrama,
            semilla_aleatoria=config.semilla_aleatoria,
        )
    
    entrenador = EntrenadorCbow(
        modelo=modelo_cbow,
        tasa_aprendizaje=config.tasa_aprendizaje,
        directorio_respaldos=config.directorio_respaldos,
        hacer_respaldo=config.hacer_respaldo,
        frecuencia_respaldo=config.frecuencia_respaldo,
    )
    
    epoca_cargada, historial_anterior = entrenador.cargar_respaldo(ruta_checkpoint_path)
    print(f"=== Checkpoint cargado: '{ruta_checkpoint_path.name}' (Época base: {epoca_cargada}) ===")
    print(f"Reanudando por +{config.cantidad_epocas} épocas adicionales...\n")
    
    historial_perdida = entrenador.entrenar_adicional(
        indices_tokens=indices_corpus,
        epocas_adicionales=config.cantidad_epocas,
        tamanio_ventana=config.tamanio_ventana,
        tamanio_lote=config.tamanio_lote,
    )

else:
    print(f"=== Iniciando Entrenamiento Limpio desde Época 0 (Motor: {config.motor_computo}) ===")
    print(f"Ventana (W): {config.tamanio_ventana} | Dimensión (N): {config.dimension_embedding} | Épocas: {config.cantidad_epocas}")
    print(f"Muestreo Negativo: {config.muestreo_negativo} | Cantidad Negativos (K): {config.cantidad_muestras_negativas}\n")
    
    if config.motor_computo == "pytorch":
        modelo_cbow = ModeloCbowPyTorch(
            tamanio_vocabulario=vocabulario.tamanio_vocabulario,
            dimension_embedding=config.dimension_embedding,
            muestreo_negativo=config.muestreo_negativo,
            cantidad_muestras_negativas=config.cantidad_muestras_negativas,
            distribucion_unigrama=distribucion_unigrama,
            semilla_aleatoria=config.semilla_aleatoria,
        )
    else:
        modelo_cbow = ModeloCbowCuPy(
            tamanio_vocabulario=vocabulario.tamanio_vocabulario,
            dimension_embedding=config.dimension_embedding,
            muestreo_negativo=config.muestreo_negativo,
            cantidad_muestras_negativas=config.cantidad_muestras_negativas,
            distribucion_unigrama=distribucion_unigrama,
            semilla_aleatoria=config.semilla_aleatoria,
        )
        
    entrenador = EntrenadorCbow(
        modelo=modelo_cbow,
        tasa_aprendizaje=config.tasa_aprendizaje,
        directorio_respaldos=config.directorio_respaldos,
        hacer_respaldo=config.hacer_respaldo,
        frecuencia_respaldo=config.frecuencia_respaldo,
    )
    
    historial_perdida = entrenador.entrenar(
        indices_tokens=indices_corpus,
        tamanio_ventana=config.tamanio_ventana,
        tamanio_lote=config.tamanio_lote,
        cantidad_epocas=config.cantidad_epocas,
        epoca_inicial=0,
    )

## Sección 5: Evaluación y Análisis del Modelo

### Subsección 5.1: Gráfico de Pérdida de Un Único Modelo
Grafica la evolución de la curva de pérdida promedio por época. Utiliza el `historial_perdida` generado en memoria durante el entrenamiento actual o permite cargar opcionalmente el historial almacenado en un archivo de respaldo especificado en la variable local `ruta_modelo_graficar`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Variable local para especificar opcionalmente una ruta de modelo guardado (.npz) a graficar.
# Dejar en None para graficar la variable de historial en memoria `historial_perdida`.
ruta_modelo_graficar = None  # Ejemplo: "respaldos/modelo_cbow_w4_epoca_1000.npz"

historial_a_graficar = []
etiqueta_grafico = "Modelo en Memoria"

if ruta_modelo_graficar is not None:
    path_archivo = Path(ruta_modelo_graficar)
    if path_archivo.exists():
        if config.motor_computo == "pytorch":
            mod_aux = ModeloCbowPyTorch(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
        else:
            mod_aux = ModeloCbowCuPy(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
        _, historial_a_graficar = mod_aux.cargar_modelo(path_archivo)
        etiqueta_grafico = path_archivo.name
        print(f"Cargado historial desde archivo: '{ruta_modelo_graficar}' ({len(historial_a_graficar)} épocas).")
    else:
        print(f"Advertencia: No existe el archivo '{ruta_modelo_graficar}'. Se usará el historial en memoria.")
        historial_a_graficar = historial_perdida
else:
    historial_a_graficar = historial_perdida

if len(historial_a_graficar) > 0:
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 5))
    epocas = list(range(1, len(historial_a_graficar) + 1))
    plt.plot(epocas, historial_a_graficar, marker="o", color="#1f77b4", linewidth=2, label=f"Pérdida ({etiqueta_grafico})")
    plt.title("Evolución de Pérdida Promedio por Época", fontsize=14, fontweight="bold")
    plt.xlabel("Época", fontsize=12)
    plt.ylabel("Pérdida Promedio", fontsize=12)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("No hay historial de pérdida disponible para graficar.")

### Subsección 5.2: Búsqueda Interactiva de Similaridad de Palabras
Evaluación de palabras similares utilizando `EvaluadorSimilaridad` sobre la variable unificada `modelo_cbow`. El parámetro `tipo_similaridad` (`"producto_interno"` o `"coseno"`) se especifica directamente en la llamada a `buscar_palabras_similares`.

In [ ]:
# Instanciar evaluador utilizando la variable global unificada modelo_cbow
evaluador = EvaluadorSimilaridad(modelo_cbow, vocabulario)

palabra_evaluar = "hombre"
top_k = 10

print(f"=== Búsqueda por PRODUCTO INTERNO para la palabra '{palabra_evaluar}' ===\n")
similares_pi = evaluador.buscar_palabras_similares(
    palabra_evaluar,
    top_k=top_k,
    tipo_similaridad="producto_interno",
)

for pos, (p, s) in enumerate(similares_pi, start=1):
    print(f"  {pos:2d}. {p:<15} Puntaje: {s:.4f}")

print(f"\n=== Búsqueda por SIMILARIDAD DE COSENO para la palabra '{palabra_evaluar}' ===\n")
similares_cos = evaluador.buscar_palabras_similares(
    palabra_evaluar,
    top_k=top_k,
    tipo_similaridad="coseno",
)

for pos, (p, s) in enumerate(similares_cos, start=1):
    print(f"  {pos:2d}. {p:<15} Coseno: {s:.4f}")

## Sección 6: Comparación entre Dos Modelos o Resguardos (.npz)
Carga y compara las curvas de pérdida y similaridad de palabras entre dos archivos de resguardo `.npz` especificando sus rutas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- RUTAS DE LOS DOS MODELOS A COMPARAR ---
ruta_modelo_1 = Path(config.directorio_respaldos) / "modelo_cbow_w4_epoca_10.npz"
ruta_modelo_2 = Path(config.directorio_respaldos) / "modelo_cbow_w5_epoca_10.npz"

print('=== Comparación de Modelos ===')
print(f'Modelo 1: {ruta_modelo_1}')
print(f'Modelo 2: {ruta_modelo_2}\n')

loss_m1 = []
loss_m2 = []

if ruta_modelo_1.exists():
    if config.motor_computo == 'pytorch':
        mod_1 = ModeloCbowPyTorch(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    else:
        mod_1 = ModeloCbowCuPy(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    ep_1, loss_m1 = mod_1.cargar_modelo(ruta_modelo_1)
else:
    print(f"El archivo {ruta_modelo_1} no existe todavía.")

if ruta_modelo_2.exists():
    if config.motor_computo == 'pytorch':
        mod_2 = ModeloCbowPyTorch(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    else:
        mod_2 = ModeloCbowCuPy(tamanio_vocabulario=vocabulario.tamanio_vocabulario, dimension_embedding=config.dimension_embedding)
    ep_2, loss_m2 = mod_2.cargar_modelo(ruta_modelo_2)
else:
    print(f"El archivo {ruta_modelo_2} no existe todavía.")

# Graficar curvas de perdida comparativas
if len(loss_m1) > 0 or len(loss_m2) > 0:
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 5))
    
    if len(loss_m1) > 0:
        epocas_m1 = []
        for idx in range(1, len(loss_m1) + 1):
            epocas_m1.append(idx)
        plt.plot(epocas_m1, loss_m1, marker="o", color="#1f77b4", linewidth=2, label=f"Modelo 1 ({ruta_modelo_1.name})")
        
    if len(loss_m2) > 0:
        epocas_m2 = []
        for idx in range(1, len(loss_m2) + 1):
            epocas_m2.append(idx)
        plt.plot(epocas_m2, loss_m2, marker="s", color="#ff7f0e", linewidth=2, label=f"Modelo 2 ({ruta_modelo_2.name})")

    plt.title("Evolución Comparativa de Pérdida Promedio por Época", fontsize=14, fontweight="bold")
    plt.xlabel("Época", fontsize=12)
    plt.ylabel("Pérdida Promedio", fontsize=12)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()